Cell 1: Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

import pandas as pd
from sentence_transformers import SentenceTransformer

from src import config
from src.chunking import build_chunks, chunk_text

df = pd.read_parquet(config.FCTR_CLEAN_PATH)
embedder = SentenceTransformer(config.EMBEDDING_MODEL)

print("Articles:", len(df))
print("Model max sequence length:", embedder.max_seq_length)

/Users/simgecevik/Desktop/WT/teyit-asistani/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1026 21:37:05.704000 5834 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11294.08it/s]


Articles: 3238
Model max sequence length: 512


Cell 2: How long are the texts in tokens?

In [2]:
def count_tokens(text):
    """Number of tokens the embedding model sees for this text."""
    return len(embedder.tokenizer(text, add_special_tokens=False)["input_ids"])

sample = df.sample(300, random_state=42)
token_lengths = sample["evidence"].map(count_tokens)

print(token_lengths.describe().round(0))
print(f"\nShare of evidence texts over 512 tokens: {(token_lengths > 512).mean():.0%}")
print(f"Characters per token (approx.): {sample['evidence'].str.len().sum() / token_lengths.sum():.1f}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2638 > 512). Running this sequence through the model will result in indexing errors


count     300.0
mean      714.0
std       563.0
min       141.0
25%       404.0
50%       582.0
75%       859.0
max      6520.0
Name: evidence, dtype: float64

Share of evidence texts over 512 tokens: 57%
Characters per token (approx.): 4.4


Cell 3: Try chunking on one article

In [3]:
amazon = df[df["doc_id"] == "fctr-2"].iloc[0]
chunks = chunk_text(amazon["evidence"], count_tokens,
                    max_tokens=config.CHUNK_MAX_TOKENS,
                    overlap_sentences=config.CHUNK_OVERLAP_SENTENCES)

print("Evidence tokens:", count_tokens(amazon["evidence"]))
print("Number of chunks:", len(chunks))
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i} ({count_tokens(chunk)} tokens) ---")
    print(chunk[:300], "...")

Evidence tokens: 1123
Number of chunks: 5

--- Chunk 0 (277 tokens) ---
Sosyal medyada sık sık yapılan paylaşımlarla Amazon’un sahiplerine ulaşamayan paketleri hediye ettiği veya cüzi miktarlarda sattığı iddia edildi. İddialara göre Amazon, teslim edilemeyen çeşitli ürünler içeren paketlerini kullanıcılara 1,95 euro karşılığında rastgele olarak dağıtıyor. Yani paketler  ...

--- Chunk 1 (275 tokens) ---
Paylaşıma eklenen bir link ile kullanıcılar 1,95 euro ve benzeri fiyatlara satılan sürpriz kutuları sipariş edebiliyor. Fakat kullanıcılar ödeme yaptıktan sonra onlara herhangi bir ürün ulaştırılmıyor, iletişime geçmek istediklerinde ise engelleniyorlar ya da cevap alamıyorlar. Yani paylaşıma eklene ...

--- Chunk 2 (281 tokens) ---
Fakat bu satış ya da bağış uygulamasını Amazon yürütmüyor. İsrafa yol açmamak adına bu sahipsiz paketler yasal yollarla ABD posta idaresi tarafından tekrar satışa çıkıyor. USPS, paketlerini 60 gün içinde teslim almayan kullanıcıların paketlerini satıyor veya

In [4]:
from src.chunking import split_sentences

for i in range(len(chunks) - 1):
    last_of_this = split_sentences(chunks[i])[-1]
    first_of_next = split_sentences(chunks[i + 1])[0]
    print(f"Chunk {i} → {i + 1}: overlap = {last_of_this == first_of_next} | {last_of_this[:60]}...")

Chunk 0 → 1: overlap = True | Paylaşıma eklenen bir link ile kullanıcılar 1,95 euro ve ben...
Chunk 1 → 2: overlap = True | Fakat bu satış ya da bağış uygulamasını Amazon yürütmüyor....
Chunk 2 → 3: overlap = True | Eğer ürün satışa çıkamayacak konumdaysa, yani defoluysa, o z...
Chunk 3 → 4: overlap = True | Dolandırıcıları ayırt etmenin bazı yöntemleri var....


Cell 4: Chunk the whole archive

In [5]:
chunks_df = build_chunks(df, count_tokens,
                         max_tokens=config.CHUNK_MAX_TOKENS,
                         overlap_sentences=config.CHUNK_OVERLAP_SENTENCES)

chunks_df["n_tokens"] = chunks_df["text"].map(count_tokens)

print("Total chunks:", len(chunks_df))
print(chunks_df["chunk_type"].value_counts())
print("\nChunks per article:")
print(chunks_df.groupby("doc_id").size().describe().round(1))
print("\nChunk token lengths:")
print(chunks_df["n_tokens"].describe().round(0))
print("\nChunks over the model limit(512):", (chunks_df["n_tokens"] > 512).sum())

Total chunks: 13242
chunk_type
evidence    10004
summary      3238
Name: count, dtype: int64

Chunks per article:
count    3238.0
mean        4.1
std         1.9
min         2.0
25%         3.0
50%         4.0
75%         5.0
max        27.0
dtype: float64

Chunk token lengths:
count    13242.0
mean       205.0
std         96.0
min         11.0
25%        105.0
50%        262.0
75%        287.0
max        594.0
Name: n_tokens, dtype: float64

Chunks over the model limit(512): 2


Cell 5: Build the index

In [6]:
import time
from src.retrieval import build_index

start = time.time()
collection = build_index(chunks_df)
print(f"\nDone in {(time.time() - start) / 60:.1f} minutes")
print("Chunks in collection:", collection.count())

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13294.25it/s]


Indexed 256 / 13242
Indexed 512 / 13242
Indexed 768 / 13242
Indexed 1024 / 13242
Indexed 1280 / 13242
Indexed 1536 / 13242
Indexed 1792 / 13242
Indexed 2048 / 13242
Indexed 2304 / 13242
Indexed 2560 / 13242
Indexed 2816 / 13242
Indexed 3072 / 13242
Indexed 3328 / 13242
Indexed 3584 / 13242
Indexed 3840 / 13242
Indexed 4096 / 13242
Indexed 4352 / 13242
Indexed 4608 / 13242
Indexed 4864 / 13242
Indexed 5120 / 13242
Indexed 5376 / 13242
Indexed 5632 / 13242
Indexed 5888 / 13242
Indexed 6144 / 13242
Indexed 6400 / 13242
Indexed 6656 / 13242
Indexed 6912 / 13242
Indexed 7168 / 13242
Indexed 7424 / 13242
Indexed 7680 / 13242
Indexed 7936 / 13242
Indexed 8192 / 13242
Indexed 8448 / 13242
Indexed 8704 / 13242
Indexed 8960 / 13242
Indexed 9216 / 13242
Indexed 9472 / 13242
Indexed 9728 / 13242
Indexed 9984 / 13242
Indexed 10240 / 13242
Indexed 10496 / 13242
Indexed 10752 / 13242
Indexed 11008 / 13242
Indexed 11264 / 13242
Indexed 11520 / 13242
Indexed 11776 / 13242
Indexed 12032 / 13242
Indexed 

In [7]:
import numpy as np

record = collection.get(ids=["fctr-2-summary"], include=["embeddings", "documents", "metadatas"])

vector = np.array(record["embeddings"][0])

print("Vector length:", len(vector))
print("Vector norm  :", round(float(np.linalg.norm(vector)), 4))
print("Metadata     :", record["metadatas"][0])
print("Text         :", record["documents"][0][:150])

Vector length: 768
Vector norm  : 1.0
Metadata     : {'verdict_group': 'FALSE', 'url': 'https://teyit.org/analiz/amazonun-turkiyede-teslim-edilmemis-urunleri-uygun-fiyata-satisa-cikardigi-iddiasi', 'chunk_type': 'summary', 'claim': "Amazon'un Türkiye'de teslim edilmemiş ürünleri uygun fiyata satışa çıkardığı iddiası", 'label': 'yanlış', 'doc_id': 'fctr-2', 'is_headline_claim': False, 'source_name': 'Teyit', 'date': '2023-07-11'}
Text         : Amazon Türkiye’de teslim edilemeyen paketlerin yeniden satışa çıkarılması gibi bir uygulama yok. ABD’de yürütülen bu uygulamada satışlar, Amazon’dan b


Cell 6: Retrieval tests

In [8]:
from src.retrieval import retrieve

def show(results):
    for r in results:
        print(f"{r['score']:.3f} | {r['doc_id']:10} | {r['chunk_type']:8} | "
              f"{r['label']:8} | {r['date']} | {r['claim'][:70]}")

print("1. Paraphrased known claim")
show(retrieve("Amazon teslim edilemeyen paketleri 2 euroya satıyormuş"))

print("\n2. Two similar articles about Threads")
show(retrieve("Threads hesabımı silersem Instagram hesabım da silinir mi?"))

print("\n3. Same Amazon query, but its own article is removed")
show(retrieve("Amazon teslim edilemeyen paketleri 2 euroya satıyormuş", exclude_doc_id="fctr-2"))

print("\n4. Probably not in the archive")
show(retrieve("Ankara'da ekim ayında kar yağdı"))

1. Paraphrased known claim
0.887 | fctr-2     | evidence | yanlış   | 2023-07-11 | Amazon'un Türkiye'de teslim edilmemiş ürünleri uygun fiyata satışa çık
0.833 | fctr-2320  | evidence | doğru    | 2020-07-10 | Amazon'da Türk bayraklı klozet takımlarının satıldığı iddiası
0.828 | fctr-820   | evidence | doğru    | 2022-09-12 | Almanya’da Lidl’ın 599 euroya Türkiye’de tatil sattığı iddiası

2. Two similar articles about Threads
0.886 | fctr-18    | evidence | doğru    | 2023-07-06 | Threads hesabını silen kullanıcıların Instagram hesabının da silineceğ
0.836 | fctr-2245  | evidence | yanlış   | 2020-08-05 | Facebook'taki kişisel verilerin kullanılmasına izin vereceği iddia edi
0.830 | fctr-2838  | evidence | yanlış   | 2019-11-01 | Facebook ve Instagram patlıcan ve şeftali emojilerinin kullanımını yas

3. Same Amazon query, but its own article is removed
0.833 | fctr-2320  | evidence | doğru    | 2020-07-10 | Amazon'da Türk bayraklı klozet takımlarının satıldığı iddiası
0.828 | fctr-820 

## Retrieval findings

- Known claims are retrieved first even when phrased differently (Amazon 0.887, Threads 0.886).
- When the correct article is excluded, the system still returns related-but-different
  articles with high scores (0.83), some labelled "doğru" → the LLM must judge relevance.
- Score gap between "correct article present" and "absent" is only ~0.05
  → a similarity threshold alone cannot separate relevant from irrelevant evidence.
- Best matches are evidence chunks describing the claim, not summaries
  → the prompt should include each article's summary as well.